# 2a · Two writers at once

Both read the same version. Both try to create the next JSON file. What happens?

In [1]:
#@title Setup — run before the session
import os, shutil, subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyspark==3.5.3', 'delta-spark==3.3.2'], check=True)
if not shutil.which('java'):
    subprocess.run('apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless', shell=True, check=True)

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from delta.tables import DeltaTable

builder = (SparkSession.builder.master('local[2]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '2')
    .config('spark.databricks.delta.properties.defaults.enableDeletionVectors', 'false')
    .config('spark.ui.showConsoleProgress', 'false'))
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('OFF')  # errors still reach Python

PATH = '/tmp/delta_talk/writers'
shutil.rmtree(PATH, ignore_errors=True)
print('Spark', spark.version, 'ready')

Spark 3.5.3 ready


In [2]:
#@title Helpers
import json, threading
SCHEMA = 'id INT, gender STRING, age INT, interest STRING'

def race(job_a, job_b):
    # Start both jobs at the same moment, report who committed, return the loser's job
    jobs, outcome = {'A': job_a, 'B': job_b}, {}
    def run(name):
        try:
            jobs[name]()
            outcome[name] = 'committed'
        except Exception as e:
            line = next(l for l in str(e).splitlines() if '[DELTA_' in l)
            outcome[name] = line[line.index(']') + 2:]
    threads = [threading.Thread(target=run, args=(n,)) for n in jobs]
    for t in threads: t.start()
    for t in threads: t.join()
    for name in jobs:
        print(f'{name}: {outcome[name]}')
    return next((jobs[n] for n in jobs if outcome[n] != 'committed'), None)

def history():
    DeltaTable.forPath(spark, PATH).history().select('version', 'operation', 'readVersion').show()

def file_report():
    added, removed = set(), set()
    for f in sorted(os.listdir(PATH + '/_delta_log')):
        if f.endswith('.json'):
            for line in open(f'{PATH}/_delta_log/{f}'):
                action = json.loads(line)
                if 'add' in action: added.add(action['add']['path'])
                if 'remove' in action: removed.add(action['remove']['path'])
    for f in sorted(f for f in os.listdir(PATH) if f.endswith('.parquet')):
        status = 'in table' if f in added - removed else 'removed by a commit' if f in removed else 'NEVER COMMITTED'
        print(f'{f[:24]}…  {status}')

## Our table, version 0

In [3]:
profiles = spark.createDataFrame(
    [(1, 'Male', 25, 'Sports'), (2, 'Female', 30, 'News'),
     (3, 'Male', 35, 'Social'), (4, 'Female', 28, 'Fashion')], SCHEMA)
profiles.coalesce(1).write.format('delta').save(PATH)
table = DeltaTable.forPath(spark, PATH)
history()

+-------+---------+-----------+
|version|operation|readVersion|
+-------+---------+-----------+
|      0|    WRITE|       NULL|
+-------+---------+-----------+



## Race 1: two jobs add new users at the same time

In [4]:
def add_user_5():
    spark.createDataFrame([(5, 'Male', 22, 'Social')], SCHEMA).coalesce(1).write.format('delta').mode('append').save(PATH)

def add_user_6():
    spark.createDataFrame([(6, 'Female', 27, 'News')], SCHEMA).coalesce(1).write.format('delta').mode('append').save(PATH)

race(add_user_5, add_user_6)

A: committed
B: committed


In [5]:
history()

+-------+---------+-----------+
|version|operation|readVersion|
+-------+---------+-----------+
|      2|    WRITE|          0|
|      1|    WRITE|          0|
|      0|    WRITE|       NULL|
+-------+---------+-----------+



## Race 2: two jobs update *different* users at the same time

In [6]:
def user_1_to_fashion():
    table.update("id = 1", {'interest': "'Fashion'"})

def user_2_to_sports():
    table.update("id = 2", {'interest': "'Sports'"})

loser = race(user_1_to_fashion, user_2_to_sports)

A: committed
B: ConcurrentAppendException: Files were added to the root of the table by a concurrent update. Please try the operation again.


In [7]:
history()

+-------+---------+-----------+
|version|operation|readVersion|
+-------+---------+-----------+
|      3|   UPDATE|          2|
|      2|    WRITE|          0|
|      1|    WRITE|          0|
|      0|    WRITE|       NULL|
+-------+---------+-----------+



## What happened to the loser's work?

In [8]:
file_report()

part-00000-44643b0a-5cb5…  in table
part-00000-64b7e687-7379…  removed by a commit
part-00000-9c34365d-19a9…  in table
part-00000-cb97cd56-9096…  in table
part-00000-ce7d2f5d-873c…  NEVER COMMITTED


## The loser just tries again

In [9]:
loser()
history()
spark.read.format('delta').load(PATH).orderBy('id').show()

+-------+---------+-----------+
|version|operation|readVersion|
+-------+---------+-----------+
|      4|   UPDATE|          3|
|      3|   UPDATE|          2|
|      2|    WRITE|          0|
|      1|    WRITE|          0|
|      0|    WRITE|       NULL|
+-------+---------+-----------+

+---+------+---+--------+
| id|gender|age|interest|
+---+------+---+--------+
|  1|  Male| 25| Fashion|
|  2|Female| 30|  Sports|
|  3|  Male| 35|  Social|
|  4|Female| 28| Fashion|
|  5|  Male| 22|  Social|
|  6|Female| 27|    News|
+---+------+---+--------+



**Users 1 and 2 are different rows. Why did their updates conflict?**